# D190 — Linux Basics for WSL/Ubuntu

A direct, lab-oriented introduction for interns. Read a short section, copy its commands into an **Ubuntu/WSL terminal**, observe the result, and then try the exercise.

> Environment used in class: Windows laptop with Ubuntu on WSL. The commands use `$USER`, `$HOME`, or `~`, so they work with each intern's own Linux account. Never put passwords in notebooks, commands, shell history, or scripts. When `sudo` prompts for a password, type it interactively; Linux shows no dots or characters while you type.

## Learning outcomes

By the end, you can navigate, manage files, read permissions such as `rwxr-x---`, search text/files, use pipes and redirection, inspect processes/resources/networking, install Ubuntu packages, locate logs, and safely use `sudo`.

## 1. Open WSL and identify the system

Open **Windows Terminal → Ubuntu**, or run `wsl` from PowerShell/Command Prompt. These commands only inspect the system.

```bash
whoami
id
hostname
uname -a
cat /etc/os-release
pwd
echo $SHELL
```

Useful help: `command --help` gives a summary, `man command` opens the manual (`q` quits), and `type command` says whether a name is a program, alias, or shell builtin.

```bash
ls --help | head
man ls
type cd
which python3
```

## 2. Linux paths and the WSL/Windows boundary

Linux uses `/` as its root. `~` and `$HOME` mean the current user's home directory, `.` means the current directory, and `..` means the parent. Paths beginning with `/` are absolute; others are relative. Linux names are case-sensitive.

Windows drives are normally mounted below `/mnt`: `C:\Course\DataEng` becomes `/mnt/c/Course/DataEng`. From WSL, `explorer.exe .` opens the current Linux location in Explorer, and `wslpath` converts paths. For Linux tools and projects, files stored under `~/...` usually behave better than files under `/mnt/c/...`.

```bash
echo $HOME
cd ~
pwd
ls /mnt/c
wslpath 'C:\Course\DataEng'
wslpath -w "$HOME"
```

## 3. Create a safe lab workspace

All file-changing examples below use this disposable directory. `mkdir -p` creates missing parent directories and does not complain if the directory already exists.

```bash
mkdir -p ~/linux_lab/{docs,data,backup,scripts}
cd ~/linux_lab
pwd
ls -la
```

Shell shortcuts: `Tab` completes names, `↑/↓` browse history, `Ctrl+C` stops the foreground command, `Ctrl+L` clears the display, and `Ctrl+D` ends input/exits a shell.

## 4. Listing and navigation

```bash
pwd                         # print working directory
ls                          # visible names
ls -l                       # long format
ls -la                      # include hidden names
ls -lh                      # human-readable sizes
ls -lt                      # newest first
cd docs                     # enter child directory
cd ..                       # go to parent
cd -                        # return to previous directory
cd ~                        # go home
```

A name beginning with `.` is hidden by convention. Quote paths containing spaces: `cd "My Folder"`.

## 5. Create, copy, move, rename, and remove

```bash
cd ~/linux_lab
touch docs/empty.txt
printf 'alpha\nbeta\ngamma\n' > docs/words.txt
cp docs/words.txt backup/words-copy.txt
cp -i docs/words.txt backup/words-copy.txt   # ask before overwrite
cp -r docs backup/docs-copy                  # copy directory tree
mv docs/empty.txt docs/renamed.txt           # rename/move
mkdir -p data/raw/2026
rmdir data/raw/2026                          # only empty directory
rm -i docs/renamed.txt                       # ask before removal
```

Treat `rm` as permanent: WSL has no automatic recycle bin. Check `pwd` and `ls` first. Avoid `rm -rf`, especially with `sudo`, variables, globs, or broad paths.

## 6. Read and edit text files

```bash
cd ~/linux_lab
cat docs/words.txt                 # entire small file
less docs/words.txt                # scroll; q quits, / searches
head -n 2 docs/words.txt
tail -n 2 docs/words.txt
nl -ba docs/words.txt              # numbered lines
file docs/words.txt                # identify file type
wc -lwc docs/words.txt             # lines, words, bytes
nano docs/notes.txt                # Ctrl+O save, Ctrl+X exit
```

For changing streams of text, `tail -f file.log` follows new lines; press `Ctrl+C` to stop.

## 7. Wildcards, quoting, variables, and history

The shell expands `*` (any string), `?` (one character), and `[abc]` (one listed character) before running a command. Always inspect a wildcard with `printf '%s\n' pattern` or `ls` before using it with `rm`/`mv`.

```bash
cd ~/linux_lab
touch data/a.csv data/b.csv data/c.txt
printf '%s\n' data/*.csv
printf '%s\n' data/?.csv
course='Linux Basics'
echo "Course: $course"              # variables expand
echo 'Course: $course'              # literal text
history | tail
history | grep 'mkdir'
```

Environment variables are inherited by child programs: `export APP_ENV=dev`; inspect with `env | sort`.

## 8. Redirection, pipes, exit status, and command chaining

A process has standard input (`stdin`, descriptor 0), output (`stdout`, 1), and error (`stderr`, 2). `>` overwrites, `>>` appends, `<` supplies input, and `|` sends one command's stdout to the next.

```bash
cd ~/linux_lab
printf 'pear\napple\npear\n' > data/fruit.txt
echo 'banana' >> data/fruit.txt
sort < data/fruit.txt | uniq -c | sort -nr
ls docs missing 1>data/output.log 2>data/error.log
ls docs missing >data/all.log 2>&1
cat data/all.log
false
echo $?                         # previous command: 0 success, nonzero failure
mkdir -p backup/out && cp data/fruit.txt backup/out/
test -f data/fruit.txt && echo 'file exists'
```

`cmd1 && cmd2` runs the second only after success; `cmd1 || cmd2` runs it only after failure; `cmd1 ; cmd2` runs both regardless.

## 9. Search files and text

```bash
cd ~/linux_lab
grep 'pear' data/fruit.txt
grep -n -i 'PEAR' data/fruit.txt       # line numbers, ignore case
grep -RIn --exclude='*.log' 'alpha' . # recursive search
find . -type f                         # files only
find . -type d                         # directories only
find . -type f -name '*.txt'
find . -type f -size +1k
find . -type f -mtime -1               # modified in last day
command -v bash
whereis bash
```

Use `find ... -print` first. Adding `-delete` changes data and should only be done after verifying the exact matches. `locate name` is faster but uses a periodically updated database and may require `sudo updatedb`.

## 10. Permissions in detail: `r`, `w`, `x`, and file flags

Run `ls -l` and read a line such as:

```text
-rwxr-x--- 1 user developers 125 Aug 11 10:00 deploy.sh
│└─user─┘└group┘└other┘        owner group      name
└ object type
```

The first character is the object type: `-` regular file, `d` directory, `l` symbolic link, `c` character device, `b` block device, `p` named pipe, and `s` socket. The next nine characters are three permission triplets: owner/user, group, and others. A `-` inside a triplet means that permission is absent.

| Flag | On a regular file | On a directory |
|---|---|---|
| `r` (read) | read contents | list names (for example, `ls`) |
| `w` (write) | change/truncate contents | create, delete, or rename entries; normally needs `x` too |
| `x` (execute) | run as a program/script | traverse/access entries (`cd`, open known names) |

Important: deleting a file is controlled by the **parent directory's** `w` and `x`, not by the file's `w`. Root can bypass many ordinary checks.

## 11. `chmod`: symbolic and octal permissions

Symbolic targets are `u` (owner), `g` (group), `o` (others), and `a` (all). Operators are `+`, `-`, and `=`. Numeric mode adds `r=4`, `w=2`, `x=1` within each triplet. Thus `7=rwx`, `6=rw-`, `5=r-x`, `4=r--`, and `0=---`.

```bash
cd ~/linux_lab
printf '#!/usr/bin/env bash\necho Hello, $USER\n' > scripts/hello.sh
ls -l scripts/hello.sh
chmod u+x scripts/hello.sh             # add owner execute
./scripts/hello.sh
chmod 750 scripts/hello.sh             # rwx r-x ---
stat scripts/hello.sh
stat -c '%A %a %U %G %n' scripts/hello.sh
chmod 640 docs/words.txt                # rw- r-- ---
```

Common modes: `600` private file, `644` ordinary non-secret file, `700` private directory/script, `755` public-readable executable/directory, `750` owner plus group access. Avoid `chmod 777`: it grants everyone full access and usually hides an ownership/design problem.

## 12. Ownership, groups, `umask`, and special bits

`chown` changes owner, `chgrp` changes group, and usually require `sudo` when assigning ownership. Do not recursively change ownership of `/`, `/etc`, `/usr`, or your Windows mount.

```bash
id
groups
ls -ln ~/linux_lab/docs/words.txt       # numeric UID/GID
sudo chown "$USER:$USER" ~/linux_lab/docs/words.txt
chgrp "$USER" ~/linux_lab/docs/words.txt
umask
umask -S
```

Default permissions start near `666` for files and `777` for directories, then `umask` removes permissions. A common `022` produces files `644` and directories `755`; `027` produces `640` and `750`.

Special leading octal digits: setuid `4` (run executable with file owner's identity), setgid `2` (on a directory, new entries inherit its group), and sticky `1` (users cannot delete each other's entries in a shared directory). Examples: `chmod 2775 shared_dir`; `/tmp` is commonly `1777` and appears as `drwxrwxrwt`. Uppercase `S`/`T` means the special bit exists while the corresponding execute bit does not. Do not set setuid/setgid casually.

## 13. Links and archives

A hard link is another directory entry for the same inode; it normally cannot cross filesystems or target a directory. A symbolic link stores a path and can cross filesystems, but can become broken.

```bash
cd ~/linux_lab
ln docs/words.txt docs/words-hardlink.txt
ln -s words.txt docs/words-link.txt
ls -li docs/words*
readlink -f docs/words-link.txt
tar -czf backup/docs.tar.gz docs/
tar -tzf backup/docs.tar.gz              # list without extracting
mkdir -p backup/restored
tar -xzf backup/docs.tar.gz -C backup/restored
du -sh backup/docs.tar.gz
```

## 14. Processes, jobs, and signals

```bash
ps
ps aux | head
pgrep -a bash
top                            # q quits
sleep 300 &                    # start background job
jobs -l
fg %1                          # foreground job 1; Ctrl+Z suspends
# bg %1                        # continue suspended job in background
# kill %1                      # politely request termination (SIGTERM)
```

Prefer `kill PID` (SIGTERM) so a program can clean up. Use `kill -9 PID` (SIGKILL) only when it will not stop normally. `pkill name` can match several processes, so check with `pgrep -a name` first. WSL may support systemd; check with `ps -p 1 -o comm=` and manage services with `systemctl status service`, `sudo systemctl start service`, and `journalctl -u service`.

## 15. CPU, memory, disks, and system information

```bash
date
uptime
lscpu | head -n 20
free -h
df -h
df -hT
du -sh ~/linux_lab
du -h --max-depth=1 ~/linux_lab | sort -h
lsblk
mount | head
```

`df` reports filesystem capacity; `du` reports space used by files/directories. WSL memory and CPU limits may reflect the WSL virtual machine configuration, not the full Windows machine.

## 16. Networking and downloads

```bash
ip addr
ip route
getent hosts example.com
ping -c 4 8.8.8.8
curl -I https://example.com
ss -tulpn
hostname -I
```

`curl -I` fetches only HTTP response headers. Download with `curl -fLO URL` or `wget URL`, but only from trusted sources. `ss -tulpn` lists listening TCP/UDP sockets; some process details need `sudo`. WSL networking differs by WSL version and Windows configuration; `localhost` integration often works, but firewall and port-forwarding rules still matter.

## 17. Users, `sudo`, and `sudo -i`

`sudo command` runs one command with elevated privileges and is preferred because the scope is visible. `sudo -i` opens a root **login shell**: the prompt usually changes from `$` to `#`, `whoami` becomes `root`, the home becomes `/root`, and root's login environment/profile is loaded. A root shell can alter or erase the entire Linux installation, so use it only when several administrative commands truly require it, then exit immediately.

```bash
whoami
sudo -v                         # refresh cached credentials
sudo whoami                    # one privileged command
sudo -l                        # show allowed sudo commands
sudo -i                        # enter root login shell
whoami
pwd
exit                           # return to your normal user; verify prompt is $
whoami
sudo -k                        # forget cached sudo credentials
```

When prompted, enter your own account password interactively. Never pipe or place a password in a command: it can leak into scripts, history, logs, or process data. Also avoid editing normal project files as root, because they may become owned by root.

## 18. Ubuntu packages with APT

```bash
apt search tree
apt show tree
sudo apt update
sudo apt install tree
tree ~/linux_lab
apt list --installed | head
sudo apt remove tree
```

`apt update` refreshes package metadata; it does not upgrade packages. `sudo apt upgrade` changes installed packages, so review its proposed changes first. Use Ubuntu repositories when possible rather than piping internet downloads into a root shell.

## 19. Where Linux and application logs are stored

Traditional persistent logs live under **`/var/log`**. On systemd-based Ubuntu, many logs are also in the binary **systemd journal**, read with `journalctl`; journal storage is commonly under `/var/log/journal` when persistent or `/run/log/journal` when volatile. Exact availability depends on the Ubuntu/WSL version, whether systemd is enabled, and each application's configuration.

| Location/command | Typical contents |
|---|---|
| `/var/log/syslog` | general Ubuntu system/service messages (when rsyslog is active) |
| `/var/log/auth.log` | authentication and `sudo` activity (when rsyslog is active) |
| `/var/log/kern.log` | kernel messages (when configured) |
| `/var/log/apt/` | APT history and terminal output |
| `/var/log/dpkg.log` | package installation/removal actions |
| `/var/log/wtmp`, `/var/log/btmp`, `/var/log/lastlog` | binary login records; read with `last`, `lastb`, `lastlog` |
| `journalctl` | systemd journal: boot, kernel, units, priorities |
| `/var/log/nginx/`, `/var/log/apache2/`, `/var/log/mysql/` | common service-specific logs when installed/configured |
| application directory/config | custom application logs; there is no universal location |

Windows-side WSL operational events can be viewed in **Event Viewer → Applications and Services Logs → Microsoft → Windows → Lxss** (provider/channel availability varies by Windows/WSL release). Windows applications may also write under `%LOCALAPPDATA%`, `%PROGRAMDATA%`, or their configured folder.

## 20. Practical log inspection

Some logs require `sudo`. Do not edit files in `/var/log`; inspect them with read-only commands. Rotated files often end in `.1` and compressed older files in `.gz` (`zless`, `zgrep`).

```bash
ls -lah /var/log
sudo less /var/log/syslog
sudo tail -n 50 /var/log/auth.log
sudo tail -f /var/log/syslog             # Ctrl+C stops following
sudo grep -i 'error' /var/log/syslog | tail
sudo zgrep -i 'error' /var/log/syslog.*.gz 2>/dev/null | tail
journalctl --list-boots
journalctl -b                            # current boot
journalctl -b -1                         # previous boot, if retained
journalctl -k                            # kernel messages
journalctl -p err..alert                 # error through alert
journalctl --since '1 hour ago'
journalctl -u ssh --since today          # one systemd unit
journalctl -f                            # follow; Ctrl+C stops
last | head
sudo lastb | head
```

If `/var/log/syslog` or `/var/log/auth.log` does not exist, that can be normal in a minimal WSL Ubuntu instance. Check `journalctl`, confirm PID 1 with `ps -p 1 -o comm=`, and inspect the relevant service's configured log destination.

## 21. Shell scripts and executable flags

The first line (`shebang`) chooses the interpreter. Quote variable expansions, use `set -euo pipefail` for many automation scripts, and validate inputs before changing data.

```bash
cd ~/linux_lab
cat > scripts/report.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
lab_dir=${1:-$HOME/linux_lab}
printf 'User: %s\n' "$USER"
printf 'Directory: %s\n' "$lab_dir"
find "$lab_dir" -maxdepth 2 -type f -printf '%p\n'
EOF
chmod 750 scripts/report.sh
bash -n scripts/report.sh                  # syntax check
./scripts/report.sh ~/linux_lab
```

`bash script.sh` only needs read permission; `./script.sh` needs execute permission and a valid shebang.

## 22. Command cheat sheet

| Goal | Commands |
|---|---|
| Identify/help | `whoami`, `id`, `uname`, `man`, `--help`, `type`, `command -v` |
| Navigate/list | `pwd`, `cd`, `ls`, `tree` |
| Files/directories | `touch`, `mkdir`, `cp`, `mv`, `rm`, `rmdir`, `ln` |
| Read/edit | `cat`, `less`, `head`, `tail`, `nano`, `file`, `wc` |
| Search/transform | `find`, `grep`, `sort`, `uniq`, `cut`, `tr`, `sed`, `awk` |
| Permissions | `ls -l`, `stat`, `chmod`, `chown`, `chgrp`, `umask` |
| Processes | `ps`, `top`, `pgrep`, `jobs`, `fg`, `bg`, `kill` |
| Storage/system | `df`, `du`, `free`, `lsblk`, `mount`, `uptime` |
| Network | `ip`, `ping`, `curl`, `wget`, `ss`, `getent` |
| Archives | `tar`, `gzip`, `gunzip`, `zip`, `unzip` |
| Administration | `sudo`, `apt`, `systemctl`, `journalctl` |

## 23. Lab verification and cleanup

Verify first; cleanup is intentionally interactive.

```bash
find ~/linux_lab -maxdepth 3 -printf '%M %u:%g %p\n' | sort
tar -tzf ~/linux_lab/backup/intern.tar.gz
bash -n ~/linux_lab/intern/bin/*.sh
# Optional cleanup after the instructor verifies your work:
rm -rI ~/linux_lab
```

`rm -rI` asks once before a recursive removal involving many entries. Read the exact target, confirm it is `$HOME/linux_lab`, and never add `sudo` for this lab.

## Key habits to keep

- Read commands before pasting them, especially commands containing `sudo`, `rm`, `chmod -R`, `chown -R`, redirection, or wildcards.
- Inspect with `pwd`, `ls`, `find`, or a dry run before making broad changes.
- Use least privilege: prefer `sudo command` to `sudo -i`; exit root shells promptly.
- Quote paths and variable expansions: `"$path"`.
- Check exit status and error output; silence errors only when you understand them.
- Store Linux projects in the Linux filesystem and use `/mnt/c` deliberately for Windows interoperability.
- Know the first places to investigate: `/var/log`, `journalctl`, the service unit, and the application's own configuration.